# CompileML governance — tamper refusal, zero-churn recalibration, certified bands

Three properties a model-risk team can verify mechanically rather than take on
faith.

In [1]:
import numpy as np

def make_credit_data(n=30_000, seed=42):
    """Synthetic credit-style dataset with a known data-generating process."""
    rng = np.random.default_rng(seed)
    utilization      = rng.beta(2, 4, n) * 1.2            # can exceed 1.0
    payment_ratio    = rng.beta(5, 2, n)                  # share of balance paid
    bills_paid_late  = rng.poisson(0.8, n).astype(float)  # late payments, 6m
    months_on_book   = rng.gamma(6, 8, n)
    n_credit_lines   = rng.poisson(4, n).astype(float) + 1
    inquiries_6m     = rng.poisson(1.2, n).astype(float)
    balance_to_limit = np.clip(utilization * rng.normal(1, 0.15, n), 0, 2)
    income_proxy     = rng.lognormal(10.5, 0.5, n) / 1e5

    logit = (
        2.2 * utilization
        - 2.6 * payment_ratio
        + 0.55 * bills_paid_late
        - 0.012 * months_on_book
        + 0.35 * inquiries_6m
        + 1.1 * balance_to_limit * (bills_paid_late > 0)   # interaction
        - 0.8 * income_proxy
        - 0.9
    )
    p_default = 1 / (1 + np.exp(-logit))
    y = (rng.random(n) < p_default).astype(int)

    X = np.column_stack([
        utilization, payment_ratio, bills_paid_late, months_on_book,
        n_credit_lines, inquiries_6m, balance_to_limit, income_proxy,
    ])
    names = [
        "UTILIZATION", "PAYMENT_RATIO", "BILLS_PAID_LATE", "MONTHS_ON_BOOK",
        "N_CREDIT_LINES", "INQUIRIES_6M", "BALANCE_TO_LIMIT", "INCOME_PROXY",
    ]
    return X, y, names

X, y, FEATURES = make_credit_data()
print(f"{X.shape[0]:,} rows, {X.shape[1]} features, default rate {y.mean():.1%}")

30,000 rows, 8 features, default rate 19.9%


In [2]:
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from compileml.compile import train_whitebox
from compileml.bands import monotone_quantile_bands
from compileml.artifact import build_artifact, save_artifact

teacher = GradientBoostingClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42
).fit(X, y)
whitebox, _ = train_whitebox(X, teacher.predict_proba(X)[:, 1],
                             n_estimators=80, random_state=42)
latent = np.clip(whitebox.predict(X), 0, 1)
artifact = build_artifact(
    whitebox, FEATURES, np.median(X, axis=0),
    monotone_quantile_bands(latent, y, n_bands=8),
    calibration_latent=latent, calibration_y=y,
)
save_artifact(artifact, "governed.json")
print("hash:", artifact["artifact_hash"])

hash: 443ad995fd31b0839c4d152bd909295bfff336a89796fcc200a7687d5c9da6c3


C:\Users\crort\AppData\Local\Temp\ipykernel_17396\3312271592.py:13: UserWarning: reason dictionary covers 0/8 features (0%). Uncovered features fall back to generic messages unsuitable for consumer-facing notices: ['UTILIZATION', 'PAYMENT_RATIO', 'BILLS_PAID_LATE', 'MONTHS_ON_BOOK', 'N_CREDIT_LINES', 'INQUIRIES_6M', 'BALANCE_TO_LIMIT', 'INCOME_PROXY']
  artifact = build_artifact(


## 1. Tampering is refused, not detected-if-you-remember-to-check

Loaders verify the SHA-256 by default. Edit one band edge and the artifact
stops loading.

In [3]:
import json
from compileml.runtime import load_artifact
from compileml.runtime.io import ArtifactError

doc = json.load(open("governed.json", encoding="utf-8"))
doc["bands"]["edges_int"][3] += 1                      # one integer, one unit
json.dump(doc, open("tampered.json", "w", encoding="utf-8"))

try:
    load_artifact("tampered.json")
except ArtifactError as exc:
    print("REFUSED:", exc)

REFUSED: artifact_hash mismatch: the document does not match its stored hash (tampered, truncated, or re-serialized non-canonically)


## 2. Zero-churn recalibration, with a hash chain instead of a promise

The portfolio drifts; PDs go stale; the model and the ladder need not move.
`recalibrate_artifact` refits the PD table on fresh outcomes while keeping the
model and edges byte-identical — so **no account changes band**, provably.

In [4]:
from compileml.artifact import recalibrate_artifact
from compileml.runtime import decide

rng = np.random.default_rng(7)
drifted_y = (rng.random(len(latent)) < np.clip(latent * 1.35, 0, 1)).astype(int)

v2 = recalibrate_artifact(artifact, latent, drifted_y)

print("model bytes identical :", v2["model"] == artifact["model"])
print("band ladder identical :", v2["bands"] == artifact["bands"])
print("new hash              :", v2["artifact_hash"][:16], "…")
print("records predecessor   :", v2["metadata"]["recalibration"]["recalibrated_from"][:16], "…")

moved, pd_old, pd_new = 0, [], []
for row in X[:2000]:
    a = decide(artifact, [float(v) for v in row], explain=False)
    b = decide(v2,       [float(v) for v in row], explain=False)
    moved += a["band"] != b["band"]
    pd_old.append(a["pd"]); pd_new.append(b["pd"])
print(f"accounts that changed band: {moved} / 2,000")
print(f"mean PD: {np.mean(pd_old):.4f} -> {np.mean(pd_new):.4f}  (drift priced in)")
assert moved == 0

model bytes identical : True
band ladder identical : True
new hash              : c016b9548c43f662 …
records predecessor   : 443ad995fd31b083 …
accounts that changed band: 0 / 2,000
mean PD: 0.1984 -> 0.2661  (drift priced in)


## 3. Certified bands that refuse to invent structure

`semantic_bands` discovers how many bands the data statistically supports —
Jeffreys-interval separation between neighbors, no residual rank power within
any band, bootstrap-certified. Give it five real risk plateaus and it finds
five; give it noise and it declines to segment.

In [5]:
from compileml.bands import semantic_bands

rng = np.random.default_rng(5)
true_pd = np.array([0.08, 0.20, 0.35, 0.55, 0.78])
levels  = np.array([0.10, 0.28, 0.46, 0.64, 0.85])
idx  = rng.integers(0, 5, 15_000)
plateau_latent = levels[idx] + rng.uniform(-0.01, 0.01, 15_000)
plateau_y      = (rng.random(15_000) < true_pd[idx]).astype(int)

found = semantic_bands(plateau_latent, plateau_y, max_bands=10, min_band_size=400,
                       eps_auc_search=0.05, eps_auc_cert=0.05, n_boot_cert=60)
print(f"discovered {found.n_bands} bands (true: 5)")
for label, pd_hat, ci in zip(found.labels, found.metadata["band_pd"],
                             found.metadata["band_pd_ci"]):
    print(f"  {label}: PD {pd_hat:.3f}  CI [{ci[0]:.3f}, {ci[1]:.3f}]")

discovered 5 bands (true: 5)
  G01: PD 0.093  CI [0.083, 0.104]
  G02: PD 0.203  CI [0.188, 0.218]
  G03: PD 0.343  CI [0.326, 0.361]
  G04: PD 0.543  CI [0.526, 0.560]
  G05: PD 0.791  CI [0.776, 0.805]


In [6]:
noise_y = rng.integers(0, 2, 15_000)
honest = semantic_bands(plateau_latent, noise_y, max_bands=10, min_band_size=400,
                        eps_auc_search=0.05, n_boot_cert=40)
print(f"on pure-noise outcomes: {honest.n_bands} band(s)")
print("flags:", honest.metadata["flags"])

on pure-noise outcomes: 1 band(s)
flags: {'no_discrete_classes': True}


The honest failure mode is the feature: a banding tool that always returns
the requested number of bands is a random number generator with labels.